# MCGPU Material File Generator

This notebook generates `.mcgpu` material files for use with the **MC-GPU** Monte Carlo
X-ray transport code (v1.3, VICTRE_MCGPU, MCGPU-PET).

It wraps `MCGPU_materials.f`, a Fortran program based on the
[PENELOPE 2006](https://www.nea.fr/html/dbprog/peneloperef.html) photon physics library.
The tool creates MFP (mean free path) tables for Rayleigh, Compton, and photoelectric
interactions, which MC-GPU uses for photon transport.

**This notebook runs in Binder or Google Colab -- no local installation required.**

---

## What you will produce

A `.mcgpu` file containing:
- Photon MFP table on a linear energy grid (5 eV steps, 5-120 keV by default)
- Rayleigh RITA sampling parameters (128 adaptive grid points)
- Compton shell structure (impulse approximation profiles)

**Reference:**  
A. Badal and A. Badano, *Med. Phys.* **36**, 4878-4880 (2009)

---
## Step 1 -- Environment setup

The cell below checks whether `gfortran` is available and compiles the generator
if the binary is not already present.  
- **Binder**: `gfortran` is pre-installed and the binary is pre-compiled via `postBuild` -- this cell will just confirm everything is ready.
- **Google Colab**: `gfortran` is installed automatically on first run (~30 seconds).

In [ ]:
import subprocess, os, sys

def run(cmd, **kw):
    """Run a shell command and print output."""
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    if r.stdout: print(r.stdout)
    if r.returncode != 0:
        print(r.stderr)
        raise RuntimeError(f"Command failed: {cmd}")
    return r

# Install gfortran if not present (needed on Colab; no-op on Binder)
if subprocess.run('which gfortran', shell=True, capture_output=True).returncode != 0:
    print("Installing gfortran...")
    run('apt-get install -y gfortran')
else:
    print("gfortran found:", subprocess.run('gfortran --version',
          shell=True, capture_output=True, text=True).stdout.splitlines()[0])

# Compile if the binary is missing (e.g. first run on Colab)
if not os.path.exists('MCGPU_materials.x'):
    print("Compiling MCGPU_materials...")
    run('gfortran MCGPU_materials.f penelope_photons.f '
        '-o MCGPU_materials.x -O3')
    print("Compilation successful.")
else:
    print("Binary MCGPU_materials.x is ready.")

# Verify PENDBASE_photons is present
n_db = len(os.listdir('PENDBASE_photons')) if os.path.isdir('PENDBASE_photons') else 0
print(f"PENDBASE_photons/: {n_db} files", "OK" if n_db >= 200 else f"WARNING: expected >=200, got {n_db}")

---
## Step 2 -- Define your material

Edit the parameters in this cell.  
The example below creates **liquid water (H2O)** -- a standard reference material
for dosimetry and tissue-equivalent simulations.

### Two ways to define a material

**Option A -- keyboard entry (used here):**  
Provide the chemical composition explicitly: atomic numbers Z and stoichiometric
indices (atoms per molecule), plus mass density.

**Option B -- PENELOPE composition database:**  
Water is also available as entry **278** in `PENDBASE_photons/pdcompos.p06`,
which contains 280 pre-defined materials (IDs 1-99 = elements, 100-280 = compounds).
To use it, set `USE_DATABASE = True` below and leave the other fields unchanged.
The database entry reads the correct composition and density automatically.

In [ ]:
# -- Material definition -------------------------------------------------------

USE_DATABASE   = False       # True  -> read from pdcompos.p06 by ID number
                             # False -> enter composition from keyboard

DATABASE_ID    = 278         # pdcompos.p06 entry for liquid water
                             # (used only when USE_DATABASE = True)

# Keyboard-entry fields (used only when USE_DATABASE = False)
MATERIAL_NAME  = 'Water, liquid'   # up to 60 characters
ELEMENTS       = [(1, 2),          # (Z, atoms/molecule): H x2
                  (8, 1)]          #                       O x1
DENSITY        = 1.0               # g/cm3
IS_INSULATOR   = True              # True  -> Fcb=Wcb=0 (correct for water,
                                   #          polymers, tissue, glass)
                                   # False -> auto plasmon (metals, semiconductors)

# -- Energy grid --------------------------------------------------------------
EMIN    = 5_000    # eV  (minimum energy)
EMAX    = 120_000  # eV  (maximum simulation energy; one extra bin above is added)
DE      = 5        # eV  (energy step; choose so (EMAX-EMIN)/DE is an integer)

# -- Output -------------------------------------------------------------------
OUTPUT_FILE = 'water_5-120keV.mcgpu'

---
## Step 3 -- Build the input stream and run the generator

`MCGPU_materials.x` is an interactive Fortran program that reads its
parameters from standard input.  The cell below builds the correct input string
from the values defined above and pipes it to the binary.

The program will print a brief log showing the Compton grouping factor and
the number of Compton shells found for this material.

In [ ]:
def build_input(use_db, db_id, name, elements, density,
                is_insulator, emin, emax, de, outfile):
    """
    Construct the standard-input stream expected by MCGPU_materials.

    Prompt order:
      1. Mode (1=keyboard, 2=database)
      2a. [database] ID number
      2b. [keyboard] name, NELEM, format (1=stoich), Z+atoms per element, density
      3. Insulator flag (1=yes, 2=no)
      4. Emin Emax (eV)
      5. DE (eV)
      6. Output filename
    """
    lines = []

    if use_db:
        lines.append('2')           # database mode
        lines.append(str(db_id))    # material ID in pdcompos.p06
    else:
        lines.append('1')           # keyboard mode
        lines.append(name)          # material name
        lines.append(str(len(elements)))  # number of elements
        if len(elements) == 1:
            lines.append(str(elements[0][0]))   # Z only for single element
        else:
            lines.append('1')       # stoichiometric format
            for z, stoich in elements:
                lines.append(f'{z} {stoich}')
        lines.append(str(density))  # mass density (g/cm3)

    lines.append('1' if is_insulator else '2')  # insulator flag
    lines.append(f'{emin} {emax}')  # energy range (eV)
    lines.append(str(de))           # energy step (eV)
    lines.append(outfile)           # output filename

    return '\n'.join(lines) + '\n'


input_str = build_input(
    USE_DATABASE, DATABASE_ID,
    MATERIAL_NAME, ELEMENTS, DENSITY,
    IS_INSULATOR, EMIN, EMAX, DE, OUTPUT_FILE
)

print("=== Input stream sent to MCGPU_materials ===")
print(input_str)

print("=== Running MCGPU_materials ===")
result = subprocess.run(
    './MCGPU_materials.x',
    input=input_str, capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise RuntimeError("Generator failed -- check the input parameters above.")

print(f"\nOutput file: {OUTPUT_FILE} ({os.path.getsize(OUTPUT_FILE):,} bytes)")

---
## Step 4 -- Inspect the output file

The `.mcgpu` file has three sections:

| Section | Content |
|---------|--------|
| MFP table | One row per energy bin: Energy, MFP_Rayleigh, MFP_Compton, MFP_Photo, MFP_Total, Rayleigh_Pmax |
| Rayleigh RITA block | 128 rows of adaptive sampling parameters for coherent scattering |
| Compton shells | One row per electron shell group (FCO, UICO, FJ0, KZCO, KSCO) |

The cell below prints the header and the Compton shell block.

In [ ]:
with open(OUTPUT_FILE) as f:
    text = f.read()

lines = text.splitlines()

# Print file header (first 10 lines)
print("=== File header ===")
print('\n'.join(lines[:10]))

# Print Compton shell block (last lines after '#[COMPTON')
compton_start = next(i for i, l in enumerate(lines) if '[COMPTON' in l)
print("\n=== Compton shell block ===")
print('\n'.join(lines[compton_start:]))

# Count data rows
mfp_rows = [l for l in lines if l and not l.startswith('#') and 'E+' in l
            and float(l.split()[0]) < 1e8]   # energy column check
print(f"\nMFP table: {len(mfp_rows)} rows")
print(f"First row: {mfp_rows[0]}")
print(f"Last row:  {mfp_rows[-1]}")

---
## Step 5 -- Plot the mean free paths

The four curves show how water interacts with photons across the diagnostic
X-ray energy range:

- **Rayleigh** (coherent): elastic scattering, dominant at very low energies
- **Compton** (incoherent): inelastic scattering, dominant above ~30 keV in water
- **Photoelectric**: dominant at low energies, shows sharp edges at shell boundaries
- **Total**: the MFP used by MC-GPU to sample the next interaction distance

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# Parse MFP table: read only the section between the MFP header and the
# Rayleigh block. The naive energy-threshold filter also captures early
# RITA rows (whose X values start near zero) and corrupts the plot.
mfp_rows = []
in_mfp = False
for l in lines:
    if '[MEAN FREE PATHS' in l and 'Energy' not in l:
        in_mfp = True
        continue
    if '[RAYLEIGH' in l:
        break
    if in_mfp and l.strip() and not l.startswith('#'):
        try:
            mfp_rows.append(list(map(float, l.split()[:6])))
        except ValueError:
            pass
data = np.array(mfp_rows)

E_keV    = data[:, 0] / 1000   # convert eV -> keV
mfp_ray  = data[:, 1]          # Rayleigh MFP (cm)
mfp_comp = data[:, 2]          # Compton MFP (cm)
mfp_phot = data[:, 3]          # Photoelectric MFP (cm)
mfp_tot  = data[:, 4]          # Total MFP (cm)

fig, ax = plt.subplots(figsize=(9, 5))

ax.semilogy(E_keV, mfp_ray,  label='Rayleigh',      color='steelblue',  lw=1.5)
ax.semilogy(E_keV, mfp_comp, label='Compton',        color='darkorange', lw=1.5)
ax.semilogy(E_keV, mfp_phot, label='Photoelectric',  color='forestgreen',lw=1.5)
ax.semilogy(E_keV, mfp_tot,  label='Total',          color='black',      lw=2.0,
            linestyle='--')

ax.set_xlabel('Photon energy (keV)', fontsize=12)
ax.set_ylabel('Mean free path (cm)', fontsize=12)
ax.set_title(f'Photon mean free paths -- {MATERIAL_NAME} (rho = {DENSITY} g/cm3)',
             fontsize=12)
ax.legend(fontsize=11)
ax.set_xlim(E_keV[0], E_keV[-1])
ax.grid(True, which='both', alpha=0.3)
ax.xaxis.set_major_formatter(ticker.FormatStrFormatter('%g'))

plt.tight_layout()
plt.savefig('mfp_plot.png', dpi=150)
plt.show()
print(f'Plot saved to mfp_plot.png  ({len(E_keV)} energy bins)')


---
## Step 6 -- Download the material file

The cell below provides a download link for the `.mcgpu` file.

In [ ]:
try:
    # Works in JupyterLab / Binder
    from IPython.display import FileLink, display
    display(FileLink(OUTPUT_FILE, result_html_prefix='Download: '))
except Exception:
    pass

try:
    # Works in Google Colab
    from google.colab import files
    files.download(OUTPUT_FILE)
except Exception:
    pass

print(f"File ready: {OUTPUT_FILE}")

---
## Other materials -- quick reference

Change the parameters in **Step 2** and re-run Steps 3-6.

### Common materials by keyboard entry

| Material | `ELEMENTS` | `DENSITY` | `IS_INSULATOR` |
|----------|-----------|-----------|----------------|
| Water H2O | `[(1,2),(8,1)]` | 1.0 | `True` |
| PMMA (acrylic) C?H?O2 | `[(6,5),(1,8),(8,2)]` | 1.19 | `True` |
| Polyethylene (CH2)? | `[(6,1),(1,2)]` | 0.94 | `True` |
| Aluminum | `[(13,1)]` | 2.699 | `False` |
| Copper | `[(29,1)]` | 8.96 | `False` |
| Tungsten | `[(74,1)]` | 19.3 | `False` |
| CsI (scintillator) | `[(55,1),(53,1)]` | 4.51 | `True` |
| Bone (cortical) | see ID 119 | 1.92 | `True` |

### Materials available in pdcompos.p06 (use `USE_DATABASE = True`)

IDs 1-99 are pure elements (Z = atomic number).  
Selected compound IDs:

| ID | Material |
|----|----------|
| 104 | Air, dry |
| 119 | Bone, cortical (ICRU) |
| 120 | Bone, compact |
| 153 | Fat, adipose tissue |
| 155 | Glandular tissue |
| 174 | Lung tissue |
| 194 | Muscle, skeletal |
| 216 | PMMA |
| 226 | Polyethylene |
| 243 | Silicon |
| 276 | Polycarbonate |
| 278 | Water, liquid |
| 279 | Water vapor |

For the full list, see `PENDBASE_photons/pdcompos.p06` (the material name appears
on the first line of each entry).

### Choosing `IS_INSULATOR`

Set `IS_INSULATOR = True` for: **all biological tissues, polymers, glass,
ceramics, and most compound materials.**  
Set `IS_INSULATOR = False` for: **pure metals and semiconductors** (Al, Cu, W,
Si, Ge, ...).

This flag controls the plasmon model for outer-shell electrons in the Compton
impulse approximation. The effect on the total MFP is negligible at diagnostic
energies but it changes the Compton shell structure written to the file, which
affects angle/energy sampling for individual Compton events in MC-GPU.

### Energy grid

The default (5 keV - 120 keV, 5 eV step) covers standard digital mammography
and chest radiography spectra. For CT or higher-energy applications:

```python
EMIN, EMAX, DE = 5_000, 200_000, 5    # up to 200 keV
EMIN, EMAX, DE = 5_000, 500_000, 10   # up to 500 keV (radiotherapy)
```

Choose `DE` such that `(EMAX - EMIN) / DE` is an integer to avoid
floating-point rounding artefacts in the energy column.